In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import mne
import plotly.express as px

In [ ]:
import sys
sys.path.insert(0, str(Path.cwd().parent / 'code'))
from compute_psd_epochs import load_all

In [ ]:
dataset = "ds003969"
tasks = ['think1', 'think2', 'med1breath', 'med2']
suffix = '_v2'
root = Path('/home/jackhoyle')
bids_root = root / 'mne_data' / dataset
processed_root = bids_root / f'processed{suffix}'

In [ ]:
# channels marked bad by process_inspect_loop.py (automated waveform + spectral LOF detection)
# are saved in info['bads'] of each .fif, which can be read without loading the data
rows = []
for fif in sorted(processed_root.glob(f'sub-*_task-*{suffix}_processed_raw.fif')):
    subject, task = re.match(r'sub-(.+)_task-(.+)'+suffix+'_processed_raw', fif.stem).groups()
    info = mne.io.read_info(fif, verbose='error')
    rows.append(dict(
        subject=subject, task=task,
        n_bad_channels=len(info['bads']),
        n_channels=len(mne.pick_types(info, eeg=True, exclude=[])),
        bad_channels=info['bads'],
    ))
bads_df = pd.DataFrame(rows)
bads_df['pct_bad_channels'] = 100 * bads_df.n_bad_channels / bads_df.n_channels
bads_df.head()

In [ ]:
# the subject/task combinations used in the analysis notebook (those with saved PSDs), and their groups
by_task, freqs, ch_names, subjects = load_all(
    bids_root / 'outputs/psd_epoch_mean',
    key='psd_mean'
)
used = pd.DataFrame([(s, t) for t, subs in subjects.items() for s in subs], columns=['subject', 'task'])

subject_info = pd.read_csv(bids_root / 'participants.tsv', sep='\t')
subject_info['subject'] = subject_info['participant_id'].str.removeprefix('sub-')

In [ ]:
df = (
    used
    .merge(bads_df, on=['subject', 'task'], how='left', validate='one_to_one')
    .merge(subject_info[['subject', 'group']], on='subject', how='left', validate='many_to_one')
)
assert df[['n_bad_channels', 'group']].notna().all().all()
df.groupby('group').subject.nunique()

In [ ]:
# mean number of bad channels per group and task
df.groupby(['task', 'group']).n_bad_channels.agg(['mean', 'median', 'max']).round(2)

In [ ]:
px.box(
    df, x='group', y='n_bad_channels', color='group', points='all', facet_col='task',
    category_orders={'task': tasks, 'group': sorted(df.group.unique())},
    hover_data=['subject', 'bad_channels'],
    labels={'n_bad_channels': 'bad channels'},
    title='Bad channels by group',
    height=450,
)

In [ ]:
# mean ± SE of the number of bad channels, per group (all tasks pooled)
summary = df.groupby('group').n_bad_channels.agg(['mean', 'sem']).reset_index()
px.bar(
    summary, x='group', y='mean', error_y='sem', color='group',
    labels={'mean': 'mean bad channels'},
    title='Mean bad channels by group (all tasks pooled)',
)